# Seed Database with Fake Data

> Populates the database with 1 year of realistic financial data using existing accounts.
>
> Generates: transactions, balance history, budgets, budget batches, and budgeted transactions.
>
> Includes nuances: recurring subscriptions, seasonal spending, occasional anomalies, income deposits, and realistic merchant variety.

In [ ]:
import pandas as pd
import numpy as np
import uuid
from datetime import date, datetime, timedelta
from dateutil.relativedelta import relativedelta
from jupyter_finance.core import db_sql, db_conn, SQL_ENGINE
from sqlalchemy import create_engine, text

## 1. Load existing accounts

In [ ]:
accounts = db_sql("SELECT account_id, name, type, subtype FROM accounts")
print(f"Found {len(accounts)} accounts")
accounts

In [ ]:
depository_ids = accounts[accounts["type"] == "depository"]["account_id"].tolist()
credit_ids = accounts[accounts["type"] == "credit"]["account_id"].tolist()
investment_ids = accounts[accounts["type"] == "investment"]["account_id"].tolist()
loan_ids = accounts[accounts["type"] == "loan"]["account_id"].tolist()

print(f"Depository: {len(depository_ids)}, Credit: {len(credit_ids)}, Investment: {len(investment_ids)}, Loan: {len(loan_ids)}")

## 2. Configuration

In [ ]:
np.random.seed(42)
TODAY = date.today()
START_DATE = TODAY - relativedelta(months=12)
NUM_DAYS = (TODAY - START_DATE).days
print(f"Generating data from {START_DATE} to {TODAY} ({NUM_DAYS} days)")

## 3. Merchant & category definitions

Realistic Canadian merchants with Plaid-style categories.

In [ ]:
RECURRING_MERCHANTS = [
    {"merchant": "Netflix", "amount": 16.49, "cat": "ENTERTAINMENT", "cat_detail": "ENTERTAINMENT_TV_AND_MOVIES", "friendly": "Discretionary", "day": 3},
    {"merchant": "Spotify", "amount": 11.99, "cat": "ENTERTAINMENT", "cat_detail": "ENTERTAINMENT_MUSIC_AND_AUDIO", "friendly": "Discretionary", "day": 7},
    {"merchant": "Rogers Wireless", "amount": 85.00, "cat": "RENT_AND_UTILITIES", "cat_detail": "RENT_AND_UTILITIES_TELEPHONE", "friendly": "Essentials", "day": 15},
    {"merchant": "Enbridge Gas", "amount": 68.50, "cat": "RENT_AND_UTILITIES", "cat_detail": "RENT_AND_UTILITIES_GAS_AND_ELECTRICITY", "friendly": "Essentials", "day": 20},
    {"merchant": "Toronto Hydro", "amount": 52.30, "cat": "RENT_AND_UTILITIES", "cat_detail": "RENT_AND_UTILITIES_GAS_AND_ELECTRICITY", "friendly": "Essentials", "day": 18},
    {"merchant": "Goodlife Fitness", "amount": 59.99, "cat": "PERSONAL_CARE", "cat_detail": "PERSONAL_CARE_GYMS_AND_FITNESS_CENTERS", "friendly": "Discretionary", "day": 1},
    {"merchant": "Adobe Creative Cloud", "amount": 72.99, "cat": "GENERAL_SERVICES", "cat_detail": "GENERAL_SERVICES_OTHER_GENERAL_SERVICES", "friendly": "Discretionary", "day": 12},
    {"merchant": "iCloud Storage", "amount": 3.99, "cat": "GENERAL_SERVICES", "cat_detail": "GENERAL_SERVICES_OTHER_GENERAL_SERVICES", "friendly": "Discretionary", "day": 25},
    {"merchant": "YouTube Premium", "amount": 13.99, "cat": "ENTERTAINMENT", "cat_detail": "ENTERTAINMENT_TV_AND_MOVIES", "friendly": "Discretionary", "day": 10},
]

RENT = {"merchant": "Mainstreet Properties", "amount": 1850.00, "cat": "RENT_AND_UTILITIES", "cat_detail": "RENT_AND_UTILITIES_RENT", "friendly": "Essentials", "day": 1}

VARIABLE_MERCHANTS = [
    # Groceries
    {"merchant": "Loblaws", "min": 35, "max": 145, "cat": "FOOD_AND_DRINK", "cat_detail": "FOOD_AND_DRINK_GROCERIES", "friendly": "Essentials", "freq": 3.5},
    {"merchant": "No Frills", "min": 25, "max": 95, "cat": "FOOD_AND_DRINK", "cat_detail": "FOOD_AND_DRINK_GROCERIES", "friendly": "Essentials", "freq": 2.0},
    {"merchant": "Costco Wholesale", "min": 80, "max": 320, "cat": "FOOD_AND_DRINK", "cat_detail": "FOOD_AND_DRINK_GROCERIES", "friendly": "Essentials", "freq": 1.5},
    {"merchant": "Farm Boy", "min": 20, "max": 75, "cat": "FOOD_AND_DRINK", "cat_detail": "FOOD_AND_DRINK_GROCERIES", "friendly": "Essentials", "freq": 1.0},
    # Restaurants & coffee
    {"merchant": "Tim Hortons", "min": 3.5, "max": 12, "cat": "FOOD_AND_DRINK", "cat_detail": "FOOD_AND_DRINK_COFFEE", "friendly": "Discretionary", "freq": 8.0},
    {"merchant": "Starbucks", "min": 5, "max": 9, "cat": "FOOD_AND_DRINK", "cat_detail": "FOOD_AND_DRINK_COFFEE", "friendly": "Discretionary", "freq": 3.0},
    {"merchant": "McDonald's", "min": 8, "max": 18, "cat": "FOOD_AND_DRINK", "cat_detail": "FOOD_AND_DRINK_FAST_FOOD", "friendly": "Discretionary", "freq": 2.0},
    {"merchant": "Uber Eats", "min": 18, "max": 55, "cat": "FOOD_AND_DRINK", "cat_detail": "FOOD_AND_DRINK_RESTAURANT", "friendly": "Discretionary", "freq": 2.5},
    {"merchant": "The Keg Steakhouse", "min": 55, "max": 140, "cat": "FOOD_AND_DRINK", "cat_detail": "FOOD_AND_DRINK_RESTAURANT", "friendly": "Discretionary", "freq": 0.5},
    # Transport
    {"merchant": "Petro-Canada", "min": 45, "max": 95, "cat": "TRANSPORTATION", "cat_detail": "TRANSPORTATION_GAS", "friendly": "Essentials", "freq": 2.0},
    {"merchant": "Shell", "min": 40, "max": 85, "cat": "TRANSPORTATION", "cat_detail": "TRANSPORTATION_GAS", "friendly": "Essentials", "freq": 1.0},
    {"merchant": "Presto Card", "min": 20, "max": 60, "cat": "TRANSPORTATION", "cat_detail": "TRANSPORTATION_PUBLIC_TRANSIT", "friendly": "Essentials", "freq": 2.0},
    {"merchant": "Uber", "min": 12, "max": 45, "cat": "TRANSPORTATION", "cat_detail": "TRANSPORTATION_TAXIS_AND_RIDE_SHARES", "friendly": "Discretionary", "freq": 1.5},
    # Shopping
    {"merchant": "Amazon.ca", "min": 15, "max": 180, "cat": "GENERAL_MERCHANDISE", "cat_detail": "GENERAL_MERCHANDISE_ONLINE_MARKETPLACES", "friendly": "Discretionary", "freq": 3.0},
    {"merchant": "Canadian Tire", "min": 12, "max": 120, "cat": "GENERAL_MERCHANDISE", "cat_detail": "GENERAL_MERCHANDISE_DEPARTMENT_STORES", "friendly": "Discretionary", "freq": 0.8},
    {"merchant": "Shoppers Drug Mart", "min": 8, "max": 55, "cat": "GENERAL_MERCHANDISE", "cat_detail": "GENERAL_MERCHANDISE_PHARMACIES", "friendly": "Essentials", "freq": 1.5},
    {"merchant": "Winners", "min": 20, "max": 90, "cat": "GENERAL_MERCHANDISE", "cat_detail": "GENERAL_MERCHANDISE_CLOTHING_AND_ACCESSORIES", "friendly": "Discretionary", "freq": 0.6},
    # Entertainment & lifestyle
    {"merchant": "Cineplex", "min": 14, "max": 35, "cat": "ENTERTAINMENT", "cat_detail": "ENTERTAINMENT_TV_AND_MOVIES", "friendly": "Discretionary", "freq": 0.5},
    {"merchant": "LCBO", "min": 15, "max": 65, "cat": "FOOD_AND_DRINK", "cat_detail": "FOOD_AND_DRINK_BEER_WINE_AND_LIQUOR", "friendly": "Discretionary", "freq": 1.5},
    # Insurance & medical
    {"merchant": "Manulife", "min": 120, "max": 120, "cat": "LOAN_PAYMENTS", "cat_detail": "LOAN_PAYMENTS_INSURANCE_PAYMENT", "friendly": "Essentials", "freq": 1.0},
]

# Seasonal spikes (holiday shopping, back to school, etc.)
SEASONAL_SPIKES = {
    12: {"label": "Holiday shopping", "merchants": ["Amazon.ca", "Hudson's Bay", "Best Buy Canada"], "multiplier": 2.5},
    11: {"label": "Black Friday", "merchants": ["Amazon.ca", "Best Buy Canada"], "multiplier": 2.0},
    9:  {"label": "Back to school", "merchants": ["Staples Canada", "Amazon.ca"], "multiplier": 1.5},
    7:  {"label": "Summer travel", "merchants": ["Air Canada", "Airbnb", "Expedia"], "multiplier": 1.0},
}

ANOMALY_MERCHANTS = [
    {"merchant": "Emergency Plumber", "amount": 850, "cat": "HOME_IMPROVEMENT", "cat_detail": "HOME_IMPROVEMENT_REPAIR_AND_MAINTENANCE", "friendly": "Essentials"},
    {"merchant": "Apple Store", "amount": 1499, "cat": "GENERAL_MERCHANDISE", "cat_detail": "GENERAL_MERCHANDISE_ELECTRONICS", "friendly": "Discretionary"},
    {"merchant": "WestJet", "amount": 680, "cat": "TRAVEL", "cat_detail": "TRAVEL_FLIGHTS", "friendly": "Discretionary"},
    {"merchant": "Auto Repair Centre", "amount": 1120, "cat": "TRANSPORTATION", "cat_detail": "TRANSPORTATION_CAR_SERVICE", "friendly": "Essentials"},
    {"merchant": "Best Buy Canada", "amount": 549, "cat": "GENERAL_MERCHANDISE", "cat_detail": "GENERAL_MERCHANDISE_ELECTRONICS", "friendly": "Discretionary"},
]

print(f"Defined {len(RECURRING_MERCHANTS)} recurring, {len(VARIABLE_MERCHANTS)} variable, {len(ANOMALY_MERCHANTS)} anomaly merchants")

## 4. Generate transactions

In [ ]:
def _uid():
    return str(uuid.uuid4())

def _jitter(amount, pct=0.05):
    """Add small random jitter to an amount."""
    return round(amount * np.random.uniform(1 - pct, 1 + pct), 2)

transactions = []

# Determine which accounts to use for spending vs income
spend_account = credit_ids[0] if credit_ids else depository_ids[0]
income_account = depository_ids[0] if depository_ids else credit_ids[0]

# --- Recurring charges (monthly, on fixed day) ---
for m in range(12):
    month_date = START_DATE + relativedelta(months=m)
    
    # Rent (always from depository / chequing)
    rent_date = month_date.replace(day=RENT["day"])
    if rent_date <= TODAY:
        transactions.append({
            "transaction_id": _uid(), "account_id": income_account,
            "amount": _jitter(RENT["amount"], 0),  # rent is exact
            "date": rent_date, "authorized_date": rent_date - timedelta(days=1),
            "merchant_name": RENT["merchant"], "name": f"RENT - {RENT['merchant']}",
            "personal_finance_category": RENT["cat"],
            "personal_finance_category_detailed": RENT["cat_detail"],
            "user_friendly_category": RENT["friendly"],
            "is_recurring": True, "is_unusual": False, "unusual_reason": None,
            "payment_channel": "online", "pending": False,
            "iso_currency_code": "CAD", "budget_run": False,
        })
    
    # Subscriptions & utilities
    for sub in RECURRING_MERCHANTS:
        try:
            txn_date = month_date.replace(day=sub["day"])
        except ValueError:
            txn_date = month_date + relativedelta(months=1) - timedelta(days=1)
        if txn_date > TODAY:
            continue
        # Occasional price increase
        amt = sub["amount"]
        if m >= 8:
            amt = round(amt * 1.03, 2)  # 3% price hike after 8 months
        transactions.append({
            "transaction_id": _uid(), "account_id": spend_account,
            "amount": _jitter(amt, 0.01),
            "date": txn_date, "authorized_date": txn_date,
            "merchant_name": sub["merchant"], "name": sub["merchant"],
            "personal_finance_category": sub["cat"],
            "personal_finance_category_detailed": sub["cat_detail"],
            "user_friendly_category": sub["friendly"],
            "is_recurring": True, "is_unusual": False, "unusual_reason": None,
            "payment_channel": "online", "pending": False,
            "iso_currency_code": "CAD", "budget_run": False,
        })

print(f"Recurring transactions: {len(transactions)}")

In [ ]:
# --- Variable / everyday spending ---
var_count_before = len(transactions)

for day_offset in range(NUM_DAYS):
    txn_date = START_DATE + timedelta(days=day_offset)
    if txn_date > TODAY:
        break
    month = txn_date.month
    is_weekend = txn_date.weekday() >= 5
    
    for merch in VARIABLE_MERCHANTS:
        # freq = average times per month; convert to daily probability
        daily_prob = merch["freq"] / 30.0
        # Slightly higher spending on weekends for food/entertainment
        if is_weekend and merch["cat"] in ("FOOD_AND_DRINK", "ENTERTAINMENT"):
            daily_prob *= 1.4
        
        if np.random.random() < daily_prob:
            amt = round(np.random.uniform(merch["min"], merch["max"]), 2)
            # Seasonal multiplier
            if month in SEASONAL_SPIKES and merch["merchant"] in SEASONAL_SPIKES[month]["merchants"]:
                amt = round(amt * SEASONAL_SPIKES[month]["multiplier"], 2)
            
            transactions.append({
                "transaction_id": _uid(), "account_id": spend_account,
                "amount": amt,
                "date": txn_date, "authorized_date": txn_date - timedelta(days=int(np.random.choice([0, 1]))),
                "merchant_name": merch["merchant"], "name": merch["merchant"],
                "personal_finance_category": merch["cat"],
                "personal_finance_category_detailed": merch["cat_detail"],
                "user_friendly_category": merch["friendly"],
                "is_recurring": False, "is_unusual": False, "unusual_reason": None,
                "payment_channel": str(np.random.choice(["in store", "online"])),
                "pending": False, "iso_currency_code": "CAD", "budget_run": False,
            })

print(f"Variable transactions: {len(transactions) - var_count_before}")

In [ ]:
# --- Seasonal one-off purchases ---
seasonal_count_before = len(transactions)

for month_num, spike in SEASONAL_SPIKES.items():
    spike_month = date(START_DATE.year if month_num >= START_DATE.month else START_DATE.year + 1, month_num, 1)
    if spike_month > TODAY:
        continue
    for merch_name in spike["merchants"]:
        if merch_name in [m["merchant"] for m in VARIABLE_MERCHANTS]:
            continue  # already handled via multiplier
        for _ in range(int(np.random.randint(1, 4))):
            txn_date = spike_month + timedelta(days=int(np.random.randint(0, 27)))
            if txn_date > TODAY:
                continue
            amt = round(np.random.uniform(50, 350) * spike["multiplier"], 2)
            transactions.append({
                "transaction_id": _uid(), "account_id": spend_account,
                "amount": amt,
                "date": txn_date, "authorized_date": txn_date,
                "merchant_name": merch_name, "name": merch_name,
                "personal_finance_category": "GENERAL_MERCHANDISE",
                "personal_finance_category_detailed": "GENERAL_MERCHANDISE_DEPARTMENT_STORES",
                "user_friendly_category": "Discretionary",
                "is_recurring": False, "is_unusual": False, "unusual_reason": None,
                "payment_channel": "online", "pending": False,
                "iso_currency_code": "CAD", "budget_run": False,
            })

print(f"Seasonal transactions: {len(transactions) - seasonal_count_before}")

In [ ]:
# --- Anomalies (large, unexpected charges scattered through the year) ---
anomaly_count_before = len(transactions)

anomaly_months = np.random.choice(range(12), size=len(ANOMALY_MERCHANTS), replace=False)
for i, anom in enumerate(ANOMALY_MERCHANTS):
    anom_date = START_DATE + relativedelta(months=int(anomaly_months[i])) + timedelta(days=int(np.random.randint(1, 28)))
    if anom_date > TODAY:
        continue
    amt = _jitter(anom["amount"], 0.1)
    transactions.append({
        "transaction_id": _uid(), "account_id": spend_account,
        "amount": amt,
        "date": anom_date, "authorized_date": anom_date,
        "merchant_name": anom["merchant"], "name": anom["merchant"],
        "personal_finance_category": anom["cat"],
        "personal_finance_category_detailed": anom["cat_detail"],
        "user_friendly_category": anom["friendly"],
        "is_recurring": False, "is_unusual": True,
        "unusual_reason": "Spike vs median",
        "payment_channel": "in store", "pending": False,
        "iso_currency_code": "CAD", "budget_run": False,
    })

print(f"Anomaly transactions: {len(transactions) - anomaly_count_before}")

In [ ]:
# --- Income (bi-weekly salary + occasional freelance) ---
income_count_before = len(transactions)
SALARY_GROSS = 3200  # bi-weekly net deposit

pay_date = START_DATE
while pay_date <= TODAY:
    # Bi-weekly salary
    transactions.append({
        "transaction_id": _uid(), "account_id": income_account,
        "amount": -SALARY_GROSS,  # negative = income in Plaid convention
        "date": pay_date, "authorized_date": pay_date,
        "merchant_name": "Employer Direct Deposit", "name": "PAYROLL - Employer Inc.",
        "personal_finance_category": "INCOME",
        "personal_finance_category_detailed": "INCOME_WAGES",
        "user_friendly_category": "Income",
        "is_recurring": True, "is_unusual": False, "unusual_reason": None,
        "payment_channel": "other", "pending": False,
        "iso_currency_code": "CAD", "budget_run": False,
    })
    pay_date += timedelta(days=14)

# Occasional freelance/side income (roughly every 2 months)
for m in range(0, 12, 2):
    fl_date = START_DATE + relativedelta(months=m) + timedelta(days=int(np.random.randint(5, 25)))
    if fl_date > TODAY:
        continue
    fl_amt = round(np.random.uniform(300, 1200), 2)
    transactions.append({
        "transaction_id": _uid(), "account_id": income_account,
        "amount": -fl_amt,
        "date": fl_date, "authorized_date": fl_date,
        "merchant_name": "E-Transfer Received", "name": "INTERAC e-Transfer",
        "personal_finance_category": "INCOME",
        "personal_finance_category_detailed": "INCOME_OTHER_INCOME",
        "user_friendly_category": "Income",
        "is_recurring": False, "is_unusual": False, "unusual_reason": None,
        "payment_channel": "other", "pending": False,
        "iso_currency_code": "CAD", "budget_run": False,
    })

# Investment contributions (monthly auto-transfer)
if investment_ids:
    for m in range(12):
        inv_date = START_DATE + relativedelta(months=m) + timedelta(days=14)
        if inv_date > TODAY:
            continue
        transactions.append({
            "transaction_id": _uid(), "account_id": investment_ids[0],
            "amount": -500,  # transfer in (negative)
            "date": inv_date, "authorized_date": inv_date,
            "merchant_name": "Wealthsimple", "name": "TFSA Contribution",
            "personal_finance_category": "TRANSFER_OUT",
            "personal_finance_category_detailed": "TRANSFER_OUT_INVESTMENT_AND_RETIREMENT_FUNDS",
            "user_friendly_category": "Savings & Investments",
            "is_recurring": True, "is_unusual": False, "unusual_reason": None,
            "payment_channel": "online", "pending": False,
            "iso_currency_code": "CAD", "budget_run": False,
        })

# Loan payment (monthly)
if loan_ids:
    for m in range(12):
        loan_date = START_DATE + relativedelta(months=m) + timedelta(days=4)
        if loan_date > TODAY:
            continue
        transactions.append({
            "transaction_id": _uid(), "account_id": loan_ids[0],
            "amount": 450.00,
            "date": loan_date, "authorized_date": loan_date,
            "merchant_name": "TD Auto Finance", "name": "Car Loan Payment",
            "personal_finance_category": "LOAN_PAYMENTS",
            "personal_finance_category_detailed": "LOAN_PAYMENTS_CAR_PAYMENT",
            "user_friendly_category": "Essentials",
            "is_recurring": True, "is_unusual": False, "unusual_reason": None,
            "payment_channel": "online", "pending": False,
            "iso_currency_code": "CAD", "budget_run": False,
        })

# --- Today's transactions (so current month shows budget consumption) ---
today_count_before = len(transactions)
TODAY_MERCHANTS = [
    ("Tim Hortons", 6.50, "FOOD_AND_DRINK", "FOOD_AND_DRINK_COFFEE", "Discretionary"),
    ("Loblaws", 72.30, "FOOD_AND_DRINK", "FOOD_AND_DRINK_GROCERIES", "Essentials"),
    ("Petro-Canada", 58.00, "TRANSPORTATION", "TRANSPORTATION_GAS", "Essentials"),
    ("Uber Eats", 34.20, "FOOD_AND_DRINK", "FOOD_AND_DRINK_RESTAURANT", "Discretionary"),
    ("Starbucks", 7.25, "FOOD_AND_DRINK", "FOOD_AND_DRINK_COFFEE", "Discretionary"),
    ("Shoppers Drug Mart", 22.99, "GENERAL_MERCHANDISE", "GENERAL_MERCHANDISE_PHARMACIES", "Essentials"),
    ("Amazon.ca", 45.00, "GENERAL_MERCHANDISE", "GENERAL_MERCHANDISE_ONLINE_MARKETPLACES", "Discretionary"),
    ("Presto Card", 50.00, "TRANSPORTATION", "TRANSPORTATION_PUBLIC_TRANSIT", "Essentials"),
]
for merchant_name, amt, cat, cat_detail, friendly in TODAY_MERCHANTS:
    transactions.append({
        "transaction_id": _uid(), "account_id": spend_account,
        "amount": amt, "date": TODAY, "authorized_date": TODAY,
        "merchant_name": merchant_name, "name": merchant_name,
        "personal_finance_category": cat, "personal_finance_category_detailed": cat_detail,
        "user_friendly_category": friendly,
        "is_recurring": False, "is_unusual": False, "unusual_reason": None,
        "payment_channel": "in store", "pending": False,
        "iso_currency_code": "CAD", "budget_run": False,
    })

print(f"Income & transfers: {len(transactions) - income_count_before}")
print(f"Today's transactions: {len(transactions) - today_count_before}")
print(f"\nTotal transactions generated: {len(transactions)}")

In [ ]:
txn_df = pd.DataFrame(transactions)
txn_df = txn_df.sort_values("date").reset_index(drop=True)

# Quick summary
print(f"Date range: {txn_df['date'].min()} to {txn_df['date'].max()}")
print(f"Total spending: ${txn_df[txn_df['amount'] > 0]['amount'].sum():,.2f}")
print(f"Total income:   ${abs(txn_df[txn_df['amount'] < 0]['amount'].sum()):,.2f}")
print(f"Unique merchants: {txn_df['merchant_name'].nunique()}")
print(f"Recurring: {txn_df['is_recurring'].sum()}, Anomalies: {txn_df['is_unusual'].sum()}")
print(f"\nMonthly spending breakdown:")
txn_df["month"] = pd.to_datetime(txn_df["date"]).dt.to_period("M")
txn_df[txn_df["amount"] > 0].groupby("month")["amount"].sum().round(2)

## 5. Generate balance history

Monthly snapshots for every account, with realistic growth for investments and declining balance for loans.

In [ ]:
balance_rows = []

for m in range(13):  # 13 snapshots = month 0 through 12
    snap_date = START_DATE + relativedelta(months=m)
    if snap_date > TODAY:
        snap_date = TODAY
    snap_dt = datetime.combine(snap_date, datetime.min.time())
    
    for _, acct in accounts.iterrows():
        aid = acct["account_id"]
        atype = acct["type"]
        
        if atype == "depository":
            # Chequing: fluctuates between 2k-8k, trending with net cash flow
            base = 4500 + np.random.uniform(-1500, 2000) + m * 80
            balance_rows.append({
                "account_id": aid, "balances_available": round(base - 200, 2),
                "balances_current": round(base, 2), "balances_iso_currency_code": "CAD",
                "balances_limit": None, "balances_unofficial_currency_code": None,
                "balances_datetime": snap_dt,
            })
        elif atype == "credit":
            # Credit card: balance between 500-3000, paid off partially each month
            bal = round(np.random.uniform(500, 3000), 2)
            balance_rows.append({
                "account_id": aid, "balances_available": round(10000 - bal, 2),
                "balances_current": bal, "balances_iso_currency_code": "CAD",
                "balances_limit": 10000.0, "balances_unofficial_currency_code": None,
                "balances_datetime": snap_dt,
            })
        elif atype == "investment":
            # TFSA/RRSP: grows from contributions + market returns
            base = 12000 + m * 550 + np.random.uniform(-400, 600)
            balance_rows.append({
                "account_id": aid, "balances_available": None,
                "balances_current": round(base, 2), "balances_iso_currency_code": "CAD",
                "balances_limit": None, "balances_unofficial_currency_code": None,
                "balances_datetime": snap_dt,
            })
        elif atype == "loan":
            # Car/student loan: decreasing balance
            base = 18000 - m * 420 + np.random.uniform(-100, 100)
            balance_rows.append({
                "account_id": aid, "balances_available": None,
                "balances_current": round(max(base, 0), 2), "balances_iso_currency_code": "CAD",
                "balances_limit": None, "balances_unofficial_currency_code": None,
                "balances_datetime": snap_dt,
            })

bal_df = pd.DataFrame(balance_rows)
print(f"Generated {len(bal_df)} balance history snapshots")
bal_df.groupby("account_id").agg({"balances_current": ["min", "max", "count"]})

## 6. Generate budgets & budget batches

In [ ]:
BUDGETS = [
    {"name": "Groceries", "description": "Monthly grocery spending", "limit": 600, "cadence": "MONTHLY", "rules": "FOOD_AND_DRINK_GROCERIES"},
    {"name": "Dining Out", "description": "Restaurants, coffee, fast food", "limit": 350, "cadence": "MONTHLY", "rules": "FOOD_AND_DRINK_RESTAURANT,FOOD_AND_DRINK_COFFEE,FOOD_AND_DRINK_FAST_FOOD"},
    {"name": "Entertainment", "description": "Subscriptions, movies, fun", "limit": 200, "cadence": "MONTHLY", "rules": "ENTERTAINMENT"},
    {"name": "Transportation", "description": "Gas, transit, rideshare", "limit": 300, "cadence": "MONTHLY", "rules": "TRANSPORTATION"},
    {"name": "Shopping", "description": "General merchandise and online", "limit": 400, "cadence": "MONTHLY", "rules": "GENERAL_MERCHANDISE"},
    {"name": "Utilities", "description": "Phone, hydro, gas, internet", "limit": 250, "cadence": "MONTHLY", "rules": "RENT_AND_UTILITIES_TELEPHONE,RENT_AND_UTILITIES_GAS_AND_ELECTRICITY"},
]

print(f"Will create {len(BUDGETS)} budgets")
for b in BUDGETS:
    print(f"  {b['name']:20s} — ${b['limit']:>7.2f}/mo — rules: {b['rules'][:50]}")

## 7. Write to database

Clears existing fake/seed data and inserts fresh records.

In [ ]:
engine = create_engine(SQL_ENGINE)

# Columns that exist in the transactions table
TXN_COLS = [
    "transaction_id", "account_id", "amount", "authorized_date", "date",
    "iso_currency_code", "merchant_name", "name", "payment_channel", "pending",
    "personal_finance_category_detailed", "personal_finance_category",
    "user_friendly_category", "is_recurring", "is_unusual", "unusual_reason", "budget_run",
]

txn_insert = txn_df[[c for c in TXN_COLS if c in txn_df.columns]].copy()
print(f"Transaction columns: {list(txn_insert.columns)}")
print(f"Rows to insert: {len(txn_insert)}")

In [ ]:
with engine.begin() as conn:
    # Clear existing data (order matters for foreign keys)
    conn.execute(text("DELETE FROM budgeted_transaction"))
    conn.execute(text("DELETE FROM budget_batch"))
    conn.execute(text("DELETE FROM budget WHERE TRUE"))
    conn.execute(text("DELETE FROM transactions"))
    conn.execute(text("DELETE FROM accounts_balance_history"))
    conn.execute(text("DELETE FROM fin_refresh"))
    print("Cleared existing data.")
    
    # Insert transactions
    txn_insert.to_sql("transactions", conn, if_exists="append", index=False)
    print(f"Inserted {len(txn_insert)} transactions.")
    
    # Insert balance history
    bal_df.to_sql("accounts_balance_history", conn, if_exists="append", index=False)
    print(f"Inserted {len(bal_df)} balance history records.")
    
    # Insert budgets
    for b in BUDGETS:
        conn.execute(text("""
            INSERT INTO budget (name, description, balance_limit, refresh_cadence, rules, is_deleted)
            VALUES (:name, :desc, :limit, :cadence, :rules, false)
        """), {"name": b["name"], "desc": b["description"], "limit": b["limit"],
               "cadence": b["cadence"], "rules": b["rules"]})
    print(f"Inserted {len(BUDGETS)} budgets.")

print("\nSeed data committed successfully.")

## 8. Create budget batches & assign transactions

For each budget, create monthly batches and assign matching transactions.

In [ ]:
budgets_db = db_sql("SELECT id, name, rules, balance_limit FROM budget WHERE is_deleted = false")
print(f"Budgets in DB: {len(budgets_db)}")
budgets_db

In [ ]:
batch_count = 0
bt_count = 0
assigned_txn_ids = set()

with engine.begin() as conn:
    for _, budget in budgets_db.iterrows():
        bid = int(budget["id"])
        rules = [r.strip() for r in str(budget["rules"]).split(",")]
        limit_val = float(budget["balance_limit"])
        
        for m in range(12):
            batch_start = START_DATE + relativedelta(months=m)
            batch_end = batch_start + relativedelta(months=1) - timedelta(days=1)
            if batch_start > TODAY:
                break
            batch_end = min(batch_end, TODAY)
            
            # Find matching transactions for this budget's rules in this month
            mask = (
                (txn_df["date"] >= batch_start) &
                (txn_df["date"] <= batch_end) &
                (txn_df["amount"] > 0) &
                (
                    txn_df["personal_finance_category"].isin(rules) |
                    txn_df["personal_finance_category_detailed"].isin(rules)
                )
            )
            matched = txn_df[mask]
            # Skip transactions already assigned to another budget
            matched = matched[~matched["transaction_id"].isin(assigned_txn_ids)]
            
            batch_total = matched["amount"].sum()
            under = batch_total <= limit_val
            # Use native Python types so psycopg2 can adapt (no numpy.bool_ / np.float64)
            bal_val = float(round(batch_total, 2))
            under_val = bool(under)
            
            # Insert budget_batch
            result = conn.execute(text("""
                INSERT INTO budget_batch (budget_id, start_date, end_date, current_balance, under_limit)
                VALUES (:bid, :start, :end, :bal, :under)
                RETURNING id
            """), {"bid": bid, "start": datetime.combine(batch_start, datetime.min.time()),
                   "end": datetime.combine(batch_end, datetime.max.time()),
                   "bal": bal_val, "under": under_val})
            batch_id = result.fetchone()[0]
            batch_count += 1
            
            # Insert budgeted_transactions
            for txn_id in matched["transaction_id"].tolist():
                conn.execute(text("""
                    INSERT INTO budgeted_transaction (batch_id, transaction_id, verified_date)
                    VALUES (:batch_id, :txn_id, :vdate)
                """), {"batch_id": batch_id, "txn_id": txn_id, "vdate": datetime.now()})
                assigned_txn_ids.add(txn_id)
                bt_count += 1
    
    # Mark budgeted transactions
    if assigned_txn_ids:
        for txn_id in assigned_txn_ids:
            conn.execute(text("UPDATE transactions SET budget_run = true WHERE transaction_id = :tid"), {"tid": txn_id})

print(f"Created {batch_count} budget batches")
print(f"Assigned {bt_count} transactions to budgets")
print(f"Updated {len(assigned_txn_ids)} transactions as budget_run = true")

## 9. Verify

In [ ]:
print("=" * 60)
print("DATABASE SEED VERIFICATION")
print("=" * 60)

counts = {
    "transactions": db_sql("SELECT COUNT(*) as n FROM transactions")["n"][0],
    "balance_history": db_sql("SELECT COUNT(*) as n FROM accounts_balance_history")["n"][0],
    "budgets": db_sql("SELECT COUNT(*) as n FROM budget WHERE is_deleted = false")["n"][0],
    "budget_batches": db_sql("SELECT COUNT(*) as n FROM budget_batch")["n"][0],
    "budgeted_txns": db_sql("SELECT COUNT(*) as n FROM budgeted_transaction")["n"][0],
}
for k, v in counts.items():
    print(f"  {k:25s}: {v:>6}")

print("\nBalance history latest per account:")
db_sql("SELECT account_id, balances_current FROM v_latest_account_balance")

In [ ]:
print("Budget overview:")
db_sql("""
    SELECT b.name, b.balance_limit,
           bb.current_balance, bb.under_limit, bb.start_date, bb.end_date
    FROM budget b
    JOIN v_latest_budget_batches bb ON bb.budget_id = b.id
    WHERE b.is_deleted = false
    ORDER BY b.name
""")

In [ ]:
print("Monthly spending by category (last 3 months):")
recent = db_sql(f"""
    SELECT date_trunc('month', date) as month,
           user_friendly_category as category,
           ROUND(SUM(amount)::numeric, 2) as total
    FROM transactions
    WHERE amount > 0 AND date >= CURRENT_DATE - INTERVAL '3 months'
    GROUP BY 1, 2
    ORDER BY 1, 3 DESC
""")
recent

In [ ]:
print("\n✅ Database seeded. Restart the Streamlit dashboard to see the data.")